# MMDL Benchmark Figure Factory

MMMU/MMMU-Pro baseline과 LLM-as-a-Judge 결과를 과제 보고용 plot으로 노트북 안에서 표시합니다.

- **raw**: 모든 문항
- **resolved-only**: rule-parser fallback 제외
- **judge-final**: judge가 판정한 fallback만 교체

성능 결과는 plot 중심으로 표시하며, 환경 metadata는 간결한 텍스트로 표시합니다.


In [ ]:
from pathlib import Path
import json, random, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "code"))
from utils import category_for_subject, official_mmmu_choice_extract, parse_options

RESULTS_ROOT = PROJECT_ROOT / "results"
INCLUDE_BACKUPS = False
plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.max_columns = 100

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


## 1. Baseline 결과

고정 결과가 존재하면 같은 결과를 복사한 legacy 날짜 디렉터리는 제외합니다.


In [ ]:
records, pro_options = [], {}

def legacy_pro_options(setting):
    if setting not in pro_options:
        config = "vision" if setting == "vision" else "standard (10 options)"
        ds = load_dataset("MMMU/MMMU_Pro", config, split="test",
                          cache_dir=PROJECT_ROOT / "datasets")
        pro_options[setting] = {
            str(x["id"]): parse_options(x["options"]) for x in ds
        }
    return pro_options[setting]

for path in sorted(RESULTS_ROOT.glob("**/predictions.jsonl")):
    parts = path.relative_to(RESULTS_ROOT).parts
    if not INCLUDE_BACKUPS and any("backup" in part.lower() for part in parts):
        continue
    exp = parts[0]
    legacy = exp == "01_mmmu_pro" and len(parts) == 5
    if legacy and (path.parent.parent / "predictions.jsonl").exists():
        continue
    if exp == "00_mmmu_vllm":
        setting, run_id, model = "validation", parts[1], parts[2]
        benchmark = "MMMU validation"
    elif exp == "01_mmmu_pro":
        setting, model = parts[1], parts[2]
        run_id = parts[3] if len(parts) == 5 else "current"
        benchmark = f"MMMU-Pro {setting}"
    else:
        continue

    label = f"{benchmark} | {model} | {run_id}"
    rng = random.Random(42)
    for row in read_jsonl(path):
        if "fallback_used" in row:
            fallback = bool(row["fallback_used"])
        elif exp == "00_mmmu_vllm":
            fallback = row.get("prediction") in ("", "Z", None)
        else:
            options = legacy_pro_options(setting)[str(row["id"])]
            _, fallback = official_mmmu_choice_extract(row["response"], options, rng)
        subject = str(row["subject"])
        records.append({
            "experiment": exp, "benchmark": benchmark, "setting": setting,
            "model": model, "run_id": run_id, "run_label": label,
            "id": str(row["id"]),
            "category": row.get("category", category_for_subject(subject)),
            "subject": subject, "correct": bool(row["correct"]),
            "fallback_used": fallback,
            "elapsed_seconds": row.get("elapsed_seconds"),
        })

baseline = pd.DataFrame(records)
if baseline.empty:
    raise FileNotFoundError("Run code/00_mmmu_vllm.py or code/01_mmmu_pro.py first.")
baseline["elapsed_seconds"] = pd.to_numeric(
    baseline["elapsed_seconds"], errors="coerce"
)


In [ ]:
run_keys = ["experiment", "benchmark", "setting", "model", "run_id", "run_label"]
run_rows = []
for keys, group in baseline.groupby(run_keys, dropna=False):
    resolved = group.loc[~group["fallback_used"]]
    timed = group["elapsed_seconds"].dropna()
    run_rows.append({
        **dict(zip(run_keys, keys)),
        "total": len(group), "raw_correct": int(group["correct"].sum()),
        "raw_accuracy": group["correct"].mean(),
        "fallback_excluded": int(group["fallback_used"].sum()),
        "resolved": len(resolved),
        "resolved_correct": int(resolved["correct"].sum()),
        "resolved_accuracy": resolved["correct"].mean() if len(resolved) else np.nan,
        "timed_samples": len(timed),
        "untimed_samples": int(group["elapsed_seconds"].isna().sum()),
        "total_elapsed_seconds": timed.sum() if len(timed) else np.nan,
        "mean_elapsed_seconds": timed.mean() if len(timed) else np.nan,
    })
runs = pd.DataFrame(run_rows).sort_values(
    ["experiment", "model", "setting", "run_id"]
).reset_index(drop=True)

# Plot the number of resolved and fallback samples for each baseline run.
fig, ax = plt.subplots(figsize=(max(9, len(runs) * 1.8), 5))
x = np.arange(len(runs))
ax.bar(x, runs["resolved"], label="Resolved by rule parser")
ax.bar(x, runs["fallback_excluded"], bottom=runs["resolved"], label="Fallback")
ax.set_xticks(x, runs["run_label"], rotation=40, ha="right")
ax.set(title="Rule-parser Coverage", ylabel="Number of samples")
ax.legend()
for index, row in runs.iterrows():
    ax.text(index, row["total"] + max(runs["total"]) * 0.01,
            f'{row["fallback_excluded"]} fallback', ha="center", fontsize=9)
fig.tight_layout()
plt.show()


## 2. LLM-as-a-Judge 결합

아직 judge 결과가 없으면 baseline 분석만 생성합니다. Judge가 성공한 문항만 교체하며 UNKNOWN은 baseline 상태로 남깁니다.


In [ ]:
judge_records = []
for path in sorted(RESULTS_ROOT.glob("02_mmmu_vllm_judge/**/judge_predictions.jsonl")):
    parts = path.relative_to(RESULTS_ROOT).parts
    for row in read_jsonl(path):
        judge_records.append({
            **row, "id": str(row["id"]),
            "source_model": parts[2], "judge_model": parts[3], "source_run_tag": parts[1],
        })
judge = pd.DataFrame(judge_records)
judge_final_frames, judge_run_rows = [], []

if not judge.empty:
    judge["elapsed_seconds"] = pd.to_numeric(judge["elapsed_seconds"], errors="coerce")
    for (source_model, judge_model, source_run_tag), judged in judge.groupby(
        ["source_model", "judge_model", "source_run_tag"]
    ):
        source = baseline.loc[
            (baseline["experiment"] == "00_mmmu_vllm")
            & (baseline["model"] == source_model)
            & (baseline["run_id"] == source_run_tag)
        ].copy()
        if source.empty:
            source = baseline.loc[
                (baseline["experiment"] == "00_mmmu_vllm")
                & (baseline["model"] == source_model)
            ].copy()
        by_id = judged.set_index("id")
        final = source.copy()
        final["judge_model"], final["judge_used"] = judge_model, False
        for idx, src in final.iterrows():
            if src["id"] in by_id.index:
                judged_row = by_id.loc[src["id"]]
                if bool(judged_row["judge_resolved"]):
                    final.at[idx, "correct"] = bool(judged_row["correct"])
                    final.at[idx, "judge_used"] = True
        judge_final_frames.append(final)
        resolved = judged.loc[judged["judge_resolved"].astype(bool)]
        fallback_n = int(source["fallback_used"].sum())
        base_correct, final_correct = int(source["correct"].sum()), int(final["correct"].sum())
        timed = judged["elapsed_seconds"].dropna()
        judge_run_rows.append({
            "source_model": source_model, "judge_model": judge_model,
            "source_total": len(source), "fallback_samples": fallback_n,
            "judged_samples": len(judged), "judge_resolved": len(resolved),
            "judge_unresolved": len(judged) - len(resolved),
            "judge_coverage": len(resolved) / fallback_n if fallback_n else np.nan,
            "recovered_correct": int(resolved["correct"].sum()),
            "baseline_accuracy": base_correct / len(source),
            "final_accuracy": final_correct / len(source),
            "accuracy_delta": (final_correct - base_correct) / len(source),
            "judge_elapsed_seconds": timed.sum() if len(timed) else np.nan,
            "judge_mean_seconds": timed.mean() if len(timed) else np.nan,
        })

judge_summary = pd.DataFrame(judge_run_rows)
if judge_summary.empty:
    print("No judge results yet. Run code/02_mmmu_vllm_judge.py.")
else:
    labels = judge_summary["judge_model"]
    x = np.arange(len(labels))
    width = 0.36
    fig, axes = plt.subplots(1, 2, figsize=(max(11, len(labels) * 3), 4.8))
    before = axes[0].bar(x - width / 2, judge_summary["baseline_accuracy"] * 100,
                         width, label="Before judge")
    after = axes[0].bar(x + width / 2, judge_summary["final_accuracy"] * 100,
                        width, label="After judge")
    axes[0].bar_label(before, fmt="%.1f")
    axes[0].bar_label(after, fmt="%.1f")
    axes[0].set_xticks(x, labels, rotation=20, ha="right")
    axes[0].set(title="Judge Accuracy Change", ylabel="Accuracy (%)", ylim=(0, 100))
    axes[0].legend()

    coverage = axes[1].bar(labels, judge_summary["judge_coverage"] * 100)
    axes[1].bar_label(coverage, fmt="%.1f%%")
    axes[1].set(title="Judge Coverage of Fallback Samples",
                ylabel="Coverage (%)", ylim=(0, 100))
    axes[1].tick_params(axis="x", rotation=20)
    fig.tight_layout()
    plt.show()


## 3. Overall accuracy 비교


In [ ]:
metric_rows, view_frames = [], []
for _, run in runs.iterrows():
    source = baseline.loc[
        (baseline["experiment"] == run["experiment"])
        & (baseline["model"] == run["model"])
        & (baseline["setting"] == run["setting"])
        & (baseline["run_id"] == run["run_id"])
    ].copy()
    for metric, frame, correct, accuracy, excluded in [
        ("raw", source, run["raw_correct"], run["raw_accuracy"], 0),
        ("resolved-only", source.loc[~source["fallback_used"]].copy(),
         run["resolved_correct"], run["resolved_accuracy"], run["fallback_excluded"]),
    ]:
        label = f'{run["run_label"]} | {metric}'
        frame["view_label"], frame["metric"] = label, metric
        view_frames.append(frame)
        metric_rows.append({
            "label": label, "benchmark": run["benchmark"], "model": run["model"],
            "metric": metric, "included": len(frame), "excluded": excluded,
            "correct": correct, "accuracy": accuracy,
        })

for final in judge_final_frames:
    source_model, judge_model = final["model"].iloc[0], final["judge_model"].iloc[0]
    label = f"MMMU validation | {source_model} + {judge_model} | judge-final"
    final = final.copy()
    final["view_label"], final["metric"] = label, "judge-final"
    view_frames.append(final)
    metric_rows.append({
        "label": label, "benchmark": "MMMU validation",
        "model": f"{source_model} + {judge_model}", "metric": "judge-final",
        "included": len(final), "excluded": 0,
        "correct": int(final["correct"].sum()), "accuracy": final["correct"].mean(),
    })

overall_metrics = pd.DataFrame(metric_rows)
analysis_samples = pd.concat(view_frames, ignore_index=True)


In [ ]:
fig, ax = plt.subplots(figsize=(max(10, len(overall_metrics) * 1.3), 6))
bars = ax.bar(overall_metrics["label"], overall_metrics["accuracy"] * 100)
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set(title="Accuracy by Evaluation View", ylabel="Accuracy (%)", ylim=(0, 100))
ax.tick_params(axis="x", rotation=50)
plt.setp(ax.get_xticklabels(), ha="right")
fig.tight_layout()
plt.show()


## 4. Category 및 Subject 성능


In [ ]:
category_metrics = (
    analysis_samples.groupby(["view_label", "metric", "category"])["correct"]
    .agg(count="size", correct="sum", accuracy="mean").reset_index()
)
subject_metrics = (
    analysis_samples.groupby(["view_label", "metric", "subject"])["correct"]
    .agg(count="size", correct="sum", accuracy="mean").reset_index()
)


In [ ]:
def accuracy_heatmap(table, index, title, filename, min_height):
    if table.empty:
        return
    matrix = table.pivot(index=index, columns="view_label", values="accuracy") * 100
    fig, ax = plt.subplots(
        figsize=(max(11, matrix.shape[1] * 1.5), max(min_height, matrix.shape[0] * 0.36))
    )
    image = ax.imshow(matrix, aspect="auto", vmin=0, vmax=100, cmap="viridis")
    ax.set_xticks(range(matrix.shape[1]), matrix.columns, rotation=50, ha="right")
    ax.set_yticks(range(matrix.shape[0]), matrix.index)
    ax.set(title=title, xlabel="Evaluation view", ylabel=index.title())
    fig.colorbar(image, ax=ax, label="Accuracy (%)")
    fig.tight_layout()
    plt.show()

accuracy_heatmap(category_metrics, "category", "Accuracy by Category",
                 "category_accuracy_heatmap.png", 5)
accuracy_heatmap(subject_metrics, "subject", "Accuracy by Subject",
                 "subject_accuracy_heatmap.png", 8)


## 5. Runtime

`elapsed_seconds`가 없는 legacy 문항은 시간 계산에서 제외하고 untimed로 표시합니다.


In [ ]:
runtime_rows = []
for (label, category, subject), group in baseline.groupby(
    ["run_label", "category", "subject"]
):
    timed = group["elapsed_seconds"].dropna()
    runtime_rows.append({
        "stage": "baseline", "run_label": label,
        "category": category, "subject": subject, "samples": len(group),
        "timed_samples": len(timed),
        "untimed_samples": int(group["elapsed_seconds"].isna().sum()),
        "total_elapsed_seconds": timed.sum() if len(timed) else np.nan,
        "mean_elapsed_seconds": timed.mean() if len(timed) else np.nan,
    })
if not judge.empty:
    for (source_model, judge_model, category, subject), group in judge.groupby(
        ["source_model", "judge_model", "category", "subject"]
    ):
        timed = group["elapsed_seconds"].dropna()
        runtime_rows.append({
            "stage": "judge", "run_label": f"{source_model} + {judge_model}",
            "category": category, "subject": subject, "samples": len(group),
            "timed_samples": len(timed),
            "untimed_samples": int(group["elapsed_seconds"].isna().sum()),
            "total_elapsed_seconds": timed.sum() if len(timed) else np.nan,
            "mean_elapsed_seconds": timed.mean() if len(timed) else np.nan,
        })
runtime_metrics = pd.DataFrame(runtime_rows)


In [ ]:
timed = runtime_metrics.dropna(subset=["total_elapsed_seconds"])
if timed.empty:
    print("No elapsed_seconds yet. Re-run experiment 00 to collect runtime.")
else:
    grouped = timed.groupby(["stage", "run_label", "category"])[
        "total_elapsed_seconds"
    ].sum().reset_index()
    grouped["label"] = grouped["stage"] + " | " + grouped["run_label"]
    matrix = grouped.pivot(index="category", columns="label",
                           values="total_elapsed_seconds")
    ax = matrix.plot.bar(figsize=(max(11, matrix.shape[1] * 1.8), 6),
                         ylabel="Elapsed seconds",
                         title="Measured Runtime by Category")
    ax.tick_params(axis="x", rotation=35)
    plt.setp(ax.get_xticklabels(), ha="right")
    ax.figure.tight_layout()
    plt.show()


## 6. Modality ablation

MMMU-Pro standard-10의 동일한 200문항에서 이미지 입력을 제거·교체한 paired pilot입니다.


In [ ]:
modality_path = (RESULTS_ROOT / "03_modality_ablation" /
                 "qwen3-vl-4b-instruct" / "predictions.jsonl")
if not modality_path.exists():
    print("No modality ablation results yet. Run code/03_modality_ablation.py.")
else:
    modality = pd.DataFrame(read_jsonl(modality_path))
    order = ["full", "text_only", "shuffled_image", "blank_image"]
    labels = ["Full", "Text only", "Shuffled image", "Blank image"]
    overall = modality.groupby("condition")["correct"].mean().reindex(order) * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bars = axes[0].bar(labels, overall, color=["#4C78A8", "#F58518", "#E45756", "#72B7B2"])
    axes[0].bar_label(bars, fmt="%.1f%%", padding=3)
    axes[0].set(title="MMMU-Pro Modality Ablation (n=200)",
                ylabel="Accuracy (%)", ylim=(0, 55))

    category = (modality.groupby(["category", "condition"])["correct"]
                .mean().unstack().reindex(columns=order) * 100)
    category.plot.bar(ax=axes[1], color=["#4C78A8", "#F58518", "#E45756", "#72B7B2"])
    axes[1].set(title="Modality Ablation by Category",
                xlabel="", ylabel="Accuracy (%)", ylim=(0, 75))
    axes[1].tick_params(axis="x", rotation=35)
    plt.setp(axes[1].get_xticklabels(), ha="right")
    axes[1].legend(labels, title="Condition")
    fig.tight_layout()
    plt.show()


## 7. Bottleneck probes

MMMU-Pro의 explanation-bearing single-image 24문항에 대한 paired 진단입니다. 작은 방향성 실험이므로 benchmark 점수로 해석하지 않습니다.


In [ ]:
probe_path = (RESULTS_ROOT / "04_bottleneck_probes" /
              "qwen3-vl-4b-instruct" / "analysis.json")
if not probe_path.exists():
    print("Run experiments 04, 05, and analyze_bottleneck_probes.py first.")
else:
    probe = json.loads(probe_path.read_text(encoding="utf-8"))
    display_order = [
        "direct", "brief_cot", "structured_reasoning", "grounded_answer",
        "low_resolution", "high_resolution", "option_permuted",
        "shuffled_no_instruction", "shuffled_with_conflict_instruction",
        "describe_then_solve", "knowledge_notes_then_solve", "oracle_rationale",
    ]
    labels = ["Direct", "Brief CoT", "Structured", "Grounded", "Low-res",
              "High-res", "Permuted", "Shuffled", "Conflict warning",
              "Describe→solve", "Knowledge→solve", "Oracle rationale"]
    raw = np.array([probe["conditions"][key]["accuracy"] * 100 for key in display_order])
    resolved = np.array([probe["conditions"][key]["accuracy_excluding_fallback"] * 100
                         for key in display_order])
    y = np.arange(len(labels))
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [1.2, 1]})
    h = 0.36
    axes[0].barh(y + h/2, raw, height=h, label="Raw", color="#4C78A8")
    axes[0].barh(y - h/2, resolved, height=h, label="Fallback excluded", color="#F58518")
    axes[0].set(yticks=y, yticklabels=labels, xlabel="Accuracy (%)",
                title="Small bottleneck probes (n=24)")
    axes[0].invert_yaxis(); axes[0].legend(); axes[0].grid(axis="x", alpha=.25)
    delta_keys = [key for key in display_order if key not in {"direct", "oracle_rationale"}]
    delta_labels = [labels[display_order.index(key)] for key in delta_keys]
    stats = [probe["paired_vs_direct"][key] for key in delta_keys]
    delta = np.array([item["delta_accuracy"] * 100 for item in stats])
    lo = np.array([item["paired_bootstrap_95_ci"][0] * 100 for item in stats])
    hi = np.array([item["paired_bootstrap_95_ci"][1] * 100 for item in stats])
    colors = ["#54A24B" if value >= 0 else "#E45756" for value in delta]
    axes[1].barh(np.arange(len(delta)), delta, color=colors, alpha=.85)
    axes[1].errorbar(delta, np.arange(len(delta)), xerr=[delta-lo, hi-delta],
                     fmt="none", ecolor="black", capsize=3)
    axes[1].axvline(0, color="black", linewidth=1)
    axes[1].set(yticks=np.arange(len(delta)), yticklabels=delta_labels,
                xlabel="Accuracy change vs direct (%p)",
                title="Paired bootstrap 95% CI")
    axes[1].invert_yaxis(); axes[1].grid(axis="x", alpha=.25)
    plt.tight_layout(); plt.show()
    stability = probe["sampling"]["all_three_predictions_same"] * 100
    print(f"Three-seed exact prediction stability: {stability:.1f}%")


## 8. 환경과 inference 설정


In [ ]:
artifact_rows = []
for path in sorted(RESULTS_ROOT.glob("**/config.json")):
    config = json.loads(path.read_text(encoding="utf-8"))
    env_path = path.parent / "environment.json"
    env = json.loads(env_path.read_text(encoding="utf-8")) if env_path.exists() else {}
    gpu, packages = env.get("gpu") or {}, env.get("packages") or {}
    artifact_rows.append({
        "result_dir": str(path.parent.relative_to(RESULTS_ROOT)),
        "benchmark": config.get("benchmark"),
        "model": config.get("model", config.get("source_model")),
        "judge_model": config.get("judge_model"),
        "backend": config.get("backend", config.get("judge_backend")),
        "max_model_length": config.get("max_model_length"),
        "max_new_tokens": config.get("max_new_tokens"),
        "temperature": config.get("temperature"), "top_p": config.get("top_p"),
        "top_k": config.get("top_k"), "min_pixels": config.get("min_pixels"),
        "max_pixels": config.get("max_pixels"), "gpu": gpu.get("name"),
        "gpu_memory_gib": gpu.get("memory_gib"), "cuda": env.get("cuda"),
        "python": env.get("python"), "torch": packages.get("torch"),
        "transformers": packages.get("transformers"),
        "datasets": packages.get("datasets"),
    })
artifacts = pd.DataFrame(artifact_rows)
if artifacts.empty:
    print("No config/environment metadata found.")
else:
    for _, row in artifacts.iterrows():
        print(f"[{row['result_dir']}]")
        print(
            f"  model={row['model']} | judge={row['judge_model']} | "
            f"backend={row['backend']} | GPU={row['gpu']} "
            f"({row['gpu_memory_gib']} GiB)"
        )
        print(
            f"  max_model_length={row['max_model_length']} | "
            f"max_new_tokens={row['max_new_tokens']} | "
            f"temperature={row['temperature']} | top_p={row['top_p']} | "
            f"top_k={row['top_k']}"
        )
        print(
            f"  Python={row['python']} | CUDA={row['cuda']} | "
            f"torch={row['torch']} | transformers={row['transformers']} | "
            f"datasets={row['datasets']}\n"
        )


## 9. 출력 정책


In [ ]:
print("Plots and metadata are displayed in the notebook only; no files are written.")
